**SCAN dataset**

[data available here](https://github.com/brendenlake/SCAN)

---



In [ ]:
import os
import re
import math
import requests

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


***downloading SCAN from GitHub***

In [ ]:
def download_github_folder(repo_owner, repo_name, folder_path, output_dir):
    """
    Download a folder from a public GitHub repo using the GitHub API.
    Saves all files (recursively) into output_dir.
    """
    api_url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/contents/{folder_path}"
    response = requests.get(api_url)
    response.raise_for_status()
    items = response.json()

    os.makedirs(output_dir, exist_ok=True)

    for item in items:
        if item["type"] == "file":
            file_url = item["download_url"]
            file_data = requests.get(file_url).content
            out_path = os.path.join(output_dir, item["name"])

            with open(out_path, "wb") as f:
                f.write(file_data)
            print(f"Downloaded file: {out_path}")

        elif item["type"] == "dir":
            # Recursively download subfolders
            download_github_folder(
                repo_owner,
                repo_name,
                item["path"],
                os.path.join(output_dir, item["name"])
            )

def download_scan_simple_split():
    """
    Convenience wrapper that downloads SCAN simple_split
    into data/simple_split.
    """
    output_dir = os.path.join("data", "simple_split")

    # Skip if already downloaded
    train_path = os.path.join(output_dir, "tasks_train_simple.txt")
    test_path  = os.path.join(output_dir, "tasks_test_simple.txt")

    if os.path.exists(train_path) and os.path.exists(test_path):
        print("SCAN simple_split already downloaded.")
        return output_dir

    print("Downloading SCAN simple_split from GitHub...")
    download_github_folder(
        repo_owner="brendenlake",
        repo_name="SCAN",
        folder_path="simple_split",
        output_dir=output_dir
    )
    return output_dir

***loading SCAN data***

In [ ]:
def load_scan_split(path):
    """
    Load SCAN tasks_*_simple.txt file.
    Each line: 'IN: ... OUT: ...'
    Returns: list of tokenized inputs, list of tokenized outputs.
    """
    inputs = []
    outputs = []

    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Extract command after 'IN: ' and before ' OUT:'
            inp = re.findall(r"IN: (.*) OUT:", line)[0]
            out = line.split("OUT: ")[1]

            inputs.append(inp.split())
            outputs.append(out.split())

    return inputs, outputs


scan_dir = download_scan_simple_split()

train_path = os.path.join(scan_dir, "tasks_train_simple.txt")
test_path  = os.path.join(scan_dir, "tasks_test_simple.txt")

train_inp, train_out = load_scan_split(train_path)
test_inp,  test_out  = load_scan_split(test_path)

print(f"Train samples: {len(train_inp)}")
print(f"Test samples:  {len(test_inp)}")

SCAN simple_split already downloaded.
Train samples: 16728
Test samples:  4182


***vocab builder***

In [ ]:
PAD = 0
BOS = 1
EOS = 2

def build_vocab(seqs):
    """
    Build mapping token -> id, including PAD/BOS/EOS.
    """
    vocab = {"<PAD>": PAD, "<BOS>": BOS, "<EOS>": EOS}
    idx = 3
    for seq in seqs:
        for tok in seq:
            if tok not in vocab:
                vocab[tok] = idx
                idx += 1
    return vocab

src_vocab = build_vocab(train_inp)
tgt_vocab = build_vocab(train_out)

src_ivocab = {v: k for k, v in src_vocab.items()}
tgt_ivocab = {v: k for k, v in tgt_vocab.items()}

print("Source vocab size:", len(src_vocab))
print("Target vocab size:", len(tgt_vocab))

Source vocab size: 16
Target vocab size: 9


***dataset & dataloader***

In [ ]:
def encode(seq, vocab):
    return [vocab[t] for t in seq]

def make_decoder_inputs(seq_ids):
    """
    seq_ids: [y1, y2, ..., yN]
    return:
      tgt_in  = [BOS, y1, ..., yN]
      tgt_out = [y1, ..., yN, EOS]
    """
    return [BOS] + seq_ids, seq_ids + [EOS]


class ScanDataset(Dataset):
    """
    Each item:
      src:     encoded source command
      tgt_in:  decoder input sequence (with BOS)
      tgt_out: decoder label sequence (with EOS)
    """
    def __init__(self, inputs, outputs, src_vocab, tgt_vocab):
        self.inputs = inputs
        self.outputs = outputs
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        src_ids = encode(self.inputs[idx], self.src_vocab)
        tgt_ids = encode(self.outputs[idx], self.tgt_vocab)
        tgt_in, tgt_out = make_decoder_inputs(tgt_ids)
        return (
            torch.tensor(src_ids, dtype=torch.long),
            torch.tensor(tgt_in, dtype=torch.long),
            torch.tensor(tgt_out, dtype=torch.long),
        )


def collate_fn(batch):
    """
    Pads variable-length src/tgt sequences into tensors of shape:
      src:     [B, max_src_len]
      tgt_in:  [B, max_tgt_len]
      tgt_out: [B, max_tgt_len]
    """
    srcs, tgt_ins, tgt_outs = zip(*batch)
    B = len(batch)

    max_src = max(len(s) for s in srcs)
    max_tgt = max(len(t) for t in tgt_ins)

    src_batch = torch.full((B, max_src), PAD, dtype=torch.long)
    tgt_in_batch = torch.full((B, max_tgt), PAD, dtype=torch.long)
    tgt_out_batch = torch.full((B, max_tgt), PAD, dtype=torch.long)

    for i in range(B):
        src_batch[i, :len(srcs[i])] = srcs[i]
        tgt_in_batch[i, :len(tgt_ins[i])] = tgt_ins[i]
        tgt_out_batch[i, :len(tgt_outs[i])] = tgt_outs[i]

    return src_batch, tgt_in_batch, tgt_out_batch


train_ds = ScanDataset(train_inp, train_out, src_vocab, tgt_vocab)
test_ds  = ScanDataset(test_inp,  test_out,  src_vocab, tgt_vocab)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True,  collate_fn=collate_fn)
test_dl  = DataLoader(test_ds,  batch_size=64, shuffle=False, collate_fn=collate_fn)

# Quick sanity check
src_b, tgt_in_b, tgt_out_b = next(iter(train_dl))
print("Batch shapes:", src_b.shape, tgt_in_b.shape, tgt_out_b.shape)

Batch shapes: torch.Size([64, 9]) torch.Size([64, 49]) torch.Size([64, 49])
